# VINE — from the question to the answer, in one run

**Press Runtime → Run all.** Nothing else to do; every cell runs in order.

For every question the whole pipeline runs:

1. **Retrieval** (`retrieve_for_compile`) collects the provisions the parser reads (Kq). No language model.
2. **The parser** (Qwen3.8-27B, effort xhigh, instructions fix5) writes the plan: the given facts, the checks, how they depend on each other, the merges.
3. **The checkers** decide each check (Claude Sonnet 5.5 with the manual lookups; the calculator and rule checker where they apply), the executor works out the merges and the decision, and the answer is written from the certificates.

**First, retrieval is measured on the 154 DEV cases** (written from the MUTCD itself, never from a test question). Retrieval was improved on 9 October 2026 (`mrag/find_provisions.py`): it now also looks the question up sentence by sentence, through the manual's own definitions (for example "QR code" → "scanning graphic", 1C.02) and through the section headings, and the ranker reads each paragraph with its Part, Section and heading. **If the new retrieval finds the target provisions for fewer DEV cases than before, the notebook stops there** and nothing else is spent.

**Questions answered:** the 11 DEV cases used before, and the 20 sample questions (text only). There is no gold answer anywhere in this notebook. The sample answers are saved for scoring outside the pipeline.

**If Colab disconnects:** press Run all again. Every stage is saved on Drive as it goes (retrieval results, every model reply, every plan, every executed question), so it carries on where it stopped.

**Before you start:**
- Runtime → Change runtime type → **A100** (40 or 80 GB).
- Your Anthropic API key in Colab's secrets (the key icon on the left) as `ANTHROPIC_API_KEY`, with access allowed for this notebook.

**Rough time and cost:** retrieval check 30–60 min; the parser about 7 min per question at xhigh (31 questions ≈ 3.5–4 hours, plus 15–25 min to install and start it); the checkers 15–30 min and about $3 of API use.

**What to send back:** the output of **CELL 5** (retrieval on DEV), **CELL 10** (plans) and **CELL 13** (DEV executed). **CELL 14** shows the sample answers and downloads them for your scoring.

In [ ]:
# CELL 1 — Drive, repo, packages.
# Data on Drive, repo on local disk. MRAG_BASE_DIR must be set BEFORE any mrag import.
import sys, os, subprocess
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"
REPO_DIR  = "/content/Beyond_RAG_repo"
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/evaluation")

# The retrieval stack, pinned as in the other notebooks. vLLM is NOT installed
# here: it gets its own environment in CELL 8, so these versions stay as they are.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0
!pip install -q -r $REPO_DIR/requirements.txt
!pip install -q --no-deps --force-reinstall "transformers>=4.49,<4.55" "huggingface_hub>=0.34,<0.35" "tokenizers>=0.21,<0.22" "torchao>=0.13,<0.14"

for f in ("mrag/find_provisions.py", "evaluation/vine_run.py", "mrag/vine/vllm_client.py",
          "mrag/vine/lookup.py", "mrag/vine/anthropic_client.py", "mrag/vine/pointers.py",
          "evaluation/parser_check.py", "evaluation/execution_check.py", "evaluation/retrieval_check.py"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"{f} is not in the clone -- push it to GitHub, then Run all again"
assert '"facts": [' in open(f"{REPO_DIR}/mrag/vine/parser.py").read(), "mrag/vine/parser.py is not fix5"
assert "def converse" in open(f"{REPO_DIR}/mrag/vine/anthropic_client.py").read(), (
    "mrag/vine/anthropic_client.py is the old version -- push the checker lookup changes first")
assert "compile_find_provisions" in open(f"{REPO_DIR}/mrag/config.py").read(), (
    "mrag/config.py has no compile_find_provisions -- push the retrieval changes first")

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("data  :", DRIVE_DIR)
print("code  :", REPO_DIR, "| commit", COMMIT)

In [ ]:
# CELL 2 — the vector store on local disk (restored from Drive if needed).
import shutil, time
from pathlib import Path
from mrag.config import CFG

LOCAL = Path(CFG.qdrant_dir)
SNAP  = Path(CFG.base_dir) / "qdrant_db.tar"

def has_store(p: Path) -> bool:
    return (p / "collection" / CFG.coll_chunks).exists()

if has_store(LOCAL):
    print("vector store already on local disk:", LOCAL)
elif SNAP.exists():
    print(f"restoring {SNAP} ({SNAP.stat().st_size / 1e6:.0f} MB) ...")
    t0 = time.time()
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    subprocess.run(["tar", "-xf", str(SNAP), "-C", str(LOCAL.parent)], check=True)
    print(f"  done in {time.time() - t0:.0f}s")
else:
    raise FileNotFoundError(f"No vector store at {LOCAL} and no snapshot at {SNAP}. "
                            "Run notebooks/Fresh_Build.ipynb first.")
assert has_store(LOCAL), f"{LOCAL}/collection/{CFG.coll_chunks} is missing after the restore"
print("ok:", LOCAL)

In [ ]:
# CELL 3 — load retrieval (text encoder, reranker, VINE graph). No LLM.
import logging, json
logging.basicConfig(level=logging.WARNING)
from mrag.ask import init_pipeline
from mrag.vine import vllm_client as vc
vc.stop_servers()    # a parser server left from a stopped run still holds the GPU (it outlives a restart)

def load_retrieval():
    p = init_pipeline(load_image_embedder=False, load_vlm=False)
    assert type(p.kg).__name__ == "VineKG", "retrieval is not on the VINE graph -- check CFG.graph_backend"
    return p

pipeline = load_retrieval()
n_store = pipeline.store.client.count(CFG.coll_chunks, exact=True).count
print(f"graph        : {pipeline.kg.g.number_of_nodes():,} nodes, {pipeline.kg.g.number_of_edges():,} edges")
print(f"vector store : {n_store:,} items")
_, _sp = pipeline.text.encode_both(["minimum mounting height of a sign"])
print(f"sparse weights in a query: {len(_sp[0])}  (0 = the sparse half of search is not working)")
SETTINGS = {k: getattr(CFG, k) for k in (
    "top_k_fused", "top_k_after_graph", "top_k_compile_chunks", "expansion_source_chunks",
    "expansion_reserved_slots", "compile_closure_budget", "compile_find_provisions",
    "compile_sentence_queries", "compile_list_depth", "compile_term_queries", "compile_term_candidates",
    "compile_heading_sections", "compile_heading_candidates", "compile_heading_paragraphs",
    "compile_pool_cap")}
print("retrieval    :", SETTINGS)
assert CFG.compile_find_provisions, "CFG.compile_find_provisions is off -- this run is for the new retrieval"

In [ ]:
# CELL 4 — the questions, and where this run keeps its results on Drive.
import importlib, hashlib
import parser_check as pc, execution_check as ec, retrieval_check as rc, vine_run as vr
for _m in (pc, ec, rc, vr):
    importlib.reload(_m)
from mrag.vine.parser import PARSER_TASK, PARSER_CONTRACT

DEV_ALL = rc.load_cases()          # 154 DEV cases (manual_cases_v2): the retrieval check
DEV = pc.load_dev_cases()          # the 11 DEV cases answered end to end (picked by a fixed rule)

# The 20 sample questions. TEXT ONLY -- no sections, answers, obligations or counts.
SAMPLE = {
 "SAMPLE002": "A conventional-road underpass has a measured minimum vertical clearance of 14 feet 7.4 inches. The statutory maximum vehicle height is 13 feet 9 inches. The location experiences frost action, and the agency has selected a 2-inch frost allowance. The clearance is uniform across the traveled way. Determine whether an advance low-clearance sign is required, the whole-inch clearance legend recommended after applying that allowance, and whether a panel mounted on the structure could replace the advance sign. If the structure panel is added, what legend should it show? Distinguish the Standard, Guidance, and Option provisions; no advance-placement distance is requested.",
 "SAMPLE003": "Use the lowest of the three high-visibility designs in Figure 3C-1. A legally established non-intersection crossing on a road posted at 40 mph is proposed with a crosswalk width of 7 feet, measured in the direction of approaching vehicle travel. Across the roadway, the design has four white longitudinal bars arranged as two identical pairs. Each bar is 10 inches wide, the clear gap inside each pair is 10 inches, and the clear gap between the two pairs is 70 inches. There are no transverse lines. The crossing location, warning signs, and other site treatments have been resolved separately. Identify the illustrated design and decide whether these specified marking dimensions and element count satisfy the applicable Standards. Give the minimum number of pairs and the permissible between-pair gap for the stated bar dimensions.\n\n**Required visual input:** [Figure 3C-1 — original PDF page 631](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=631). Read the specified drawing; its geometry is not to scale.",
 "SAMPLE004": "Two entrance-ramp alternatives are being checked for signal-face count and placement only. In Alternative A, the ramp has one metered general-purpose lane and a non-metered HOV bypass lane. Two faces serve the metered lane, both on one roadside pole, with the lower face mounted 4.5 feet high. In Alternative B, both lanes are metered and their greens alternate rather than always appearing simultaneously. B provides exactly one face for each lane, each on the nearest roadside pole, with no face over either lane. Other signal design and visibility provisions have been satisfied. Can the count-and-placement arrangements be accepted in both alternatives? Explain how the controlled-lane count changes the applicable rule and identify a compliant correction to B.",
 "SAMPLE005": "A reversible-lane installation, away from any toll plaza, has minimal visual clutter and is a typical application. During some periods the lane serves left turns, and all required arrow and X indications have been provided. The nominal face dimensions are 12 by 12 inches for the downward green arrow, yellow X, and red X, and 18 by 18 inches for each applicable white left-turn arrow. Check only these face dimensions. In Operating Case A, the posted and statutory speed limits, 85th-percentile speed, and operating speed are all 35 mph. In Case B all four are 40 mph. Can the same small faces be accepted in both cases? Explain the exception boundary and the nominal minimum dimensions that apply when it is unavailable.",
 "SAMPLE006": "An ordinary signalized crosswalk has a concurrent vehicular phase, with an engineered pedestrian sequence of 7 seconds WALK, 12 seconds flashing UPRAISED HAND, and 3 seconds steady UPRAISED HAND as a buffer. No preemption occurs. A pedestrian enters the crosswalk 110 feet from the countdown display. The proposed display uses 6-inch Portland orange numbers on a black opaque background immediately adjacent to the hand indication. It stays dark during WALK, begins at 15 when the flashing hand starts, and continues counting through the 3-second steady-hand buffer before reaching zero. Are its countdown operation and numeral height acceptable under their respective authorities? State what time should be counted and whether the long viewing distance creates a mandatory or recommended larger numeral size. Do not recalculate the engineered pedestrian intervals.",
 "SAMPLE007": "A two-lane, one-way public highway crosses two railroad tracks at a passive crossing. There are no active warning devices and no authorized person routinely stopping road users when a train approaches. There is no nearby signalized highway intersection, and no engineering study has justified replacing the default YIELD control with STOP control. The proposal provides only a right-side Crossbuck with a 2 TRACKS plaque and a YIELD sign on a separate support. The YIELD sign positioning and height, Crossbuck blade retroreflection, and other placement details have already been checked. Both the front and back of the Crossbuck support are left without retroreflective strips, on the claim that one-way operation permits both omissions. Are the assembly count, control choice, and support-strip omissions acceptable? State what must change and which permission can still be used.",
 "SAMPLE008": "A fabricator is preparing a paddle for an adult school crossing guard who will use it before dawn. One face says STOP; the other copies the face shown in the middle row of the PREFERRED METHOD column in Figure 6D-1. The paddle is octagonal, 24 inches across, on a rigid handle, and fully retroreflectorized. All other size and legend details are appropriate to the face being copied. The STOP face has two flashing lights, one centered above its legend and one below: the upper light is white, the lower red, and they flash at 55 flashes per minute. Can this device be accepted for the school guard? Identify the copied face, follow the school-use cross-reference, and distinguish the permitted flashing-light feature from its mandatory restrictions.\n\n**Required visual input:** [Figure 6D-1 — original PDF page 825](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=825). Read the specified drawing; its geometry is not to scale.",
 "SAMPLE009": "An emergency-vehicle hybrid beacon has been justified for an unsignalized fire-station access. All application criteria have been satisfied, no STOP- or YIELD-controlled side-road intersection lies within 100 feet, and no grade crossing is involved. It is actuated only by authorized personnel. On each major-street approach, all relevant speed measures are 45 mph. There are two beacon faces per approach, both roadside and none over the roadway; required signs and stop lines are provided. The normal program is dark, flashing yellow, steady yellow, alternating flashing red for sufficient vehicle egress, then dark again. The yellow timing has been properly engineered. If the conflict monitor puts the beacon into flashing mode, the proposed fault program also displays alternating flashing red. Assess face count, recommended placement, normal operation, and fault operation. Which change is mandatory and which is a Guidance matter?",
 "SAMPLE010": "A three-lane, one-way street has a marked midblock crosswalk across an uncontrolled approach. An engineering study supports warning lights embedded in the pavement, applicable warning signs are installed, and no STOP, YIELD, traffic signal, or pedestrian hybrid beacon controls the crossing. The approach-side installation extends across the full crossing and provides three unidirectional units, one per lane. Each is 9 feet from the outside edge of the nearest crosswalk line and faces away from the crosswalk toward approaching traffic. No units are proposed on the departure side. Passive pedestrian detection starts flashing and ends operation after the pedestrian clears the full crosswalk. Yellow output has 55 flash periods per minute; within those periods, its separate flicker modulation repeats at 4 pulses per second. Units project 0.5 inch above the pavement. Evaluate only side-of-crosswalk placement, minimum count, offset, facing direction, actuation, flash/pulse rates, and height. Can those specified features be accepted, and which are governed by an Option or Guidance rather than a Standard?",
 "SAMPLE011": "A continuous paved, undivided, two-way arterial has exactly two traffic lanes, a 20-foot traveled way, and an ADT of 6,500 vehicles. One segment is classified rural and the adjoining segment urban. It is neither a freeway nor an expressway. An engineering study also identifies a need for edge lines on the urban segment. There is no finding that edge lines would decrease safety, and no edge-line exclusion or omission is being used. Considering only whether center lines and edge lines are called for, identify which are required by a Standard and which are recommended by Guidance on each segment. A reviewer claims that unchanged width and traffic volume make the authority identical on both segments. Is that claim correct?",
 "SAMPLE012": "An engineering study of a three-leg public-road intersection identifies five reported crashes during a 36-month period, all of types susceptible to correction by all-way stop control. No 12-month period within that record contains more than three such crashes. The study evaluates the other relevant site factors and supports all-way stop control; the agency has decided to install STOP signs on all three approaches. A reviewer says the crash count alone makes installation compulsory and proposes a 3-WAY plaque below every STOP sign instead of an ALL-WAY plaque. Does the crash record meet the applicable crash warrant, does that warrant alone compel installation, and does the proposed plaque scheme satisfy the signing requirements once all-way stop control is selected?",
 "SAMPLE013": "A two-lane, two-way roadway already has center line markings. Its posted speed limit, statutory speed limit, and 85th-percentile speed are all 45 mph. At two horizontal curves, an engineering study documents passing sight distances of 650 feet and establishes the limits of the sight-distance-restricted areas for one direction as stations 10+00 to 15+00 and 18+50 to 22+00. The intervening section has adequate passing sight distance; no other condition changes these boundaries. Stationing is in feet. Determine whether the curve zones require no-passing markings, whether the zones should be joined across the gap, and the total continuous marked length if the joining recommendation is followed. Distinguish the mandatory marking rule from the short-gap recommendation.",
 "SAMPLE014": "An agency will replace a broken white line with raised pavement markers, rather than merely supplementing the painted line. The underlying 20-foot line segment and 40-foot gap pattern has already been justified; do not reassess that pattern selection. The proposed replacement uses three non-retroreflective markers at the start, midpoint, and end of each 20-foot line segment, plus one retroreflective marker exactly halfway along every 40-foot gap between groups. Marker colors and other unrelated design details are appropriate. Is the three-marker group acceptable? Determine the smallest permitted group count and its equal spacing that can span each 20-foot segment while meeting the spacing limit. Must a retroreflective marker be inside each group, or can the gap-midpoint markers satisfy that requirement?",
 "SAMPLE015": "A stationary freeway work zone closes the two rightmost lanes of a four-lane roadway. Traffic must merge left through two properly designed successive merging tapers; there is no open route to the right of the closure. The contractor provides just one Type C arrow board and operates it in the display shown under operating mode 2 in Figure 6L-3, claiming that its two arrowheads cover the two closed lanes. The board is 96 by 48 inches, has 15 elements, and has a verified legibility distance of 3/4 mile. It has all required mode capabilities, including a left-arrow mode, can dim by 50 percent from full brilliance, and uses that dimmed setting at night. Other device details have been checked. Interpret the selected display, determine the required board count, and assess the stated Type C dimensions, element count, legibility, and dimming. Can this proposal be accepted for those checks?\n\n**Required visual input:** [Figure 6L-3 — original PDF page 875](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=875). Read the specified original diagram; do not infer dimensions by scaling it.",
 "SAMPLE016": "A one-lane, two-way temporary traffic control zone uses a red/yellow-lens Automated Flagger Assistance Device (AFAD) at each end, with only one approach lane in each direction. One trained flagger operates both devices. The flagger has an unobstructed view of both AFADs, but a curve blocks the view of approaching traffic in one direction. There is no second flagger. To stop traffic, each device changes from flashing yellow to a 5-second steady yellow and then steady red; its gate begins descending during the steady-yellow interval. To release traffic, it changes from steady red to a 5-second steady yellow and then flashing yellow. Required safeguards against simultaneous releases, clearance of opposing vehicles, and other device details are satisfied. Assess the one-flagger exception and both transitions. Does a 5-second yellow duration make the operation acceptable? Distinguish the duration recommendation from the mandatory operating rules.",
 "SAMPLE017": "Two temporary rumble-strip treatments are proposed on black pavement on roads used by bicyclists. Neither location is a bicycle route or a pedestrian crossing, and other siting and skid-resistance considerations have been addressed. Treatment A consists of orange transverse rumble strips within the travel lanes, with a clear path exactly 4 feet wide at each edge of the roadway. Treatment B consists of white longitudinal rumble strips on the shoulder supplementing a white right-hand edge line; it leaves a clear shoulder path 3.5 feet wide. Do both color choices satisfy the mandatory color provisions, and do both bicycle clearances meet the recommendations? May the orange-color permission for A simply be carried over to B while B continues to supplement a white edge line?",
 "SAMPLE018": "Queues from a nearby traffic signal regularly stop across an upstream unsignalized intersection. The agency decides to use Do Not Block Intersection markings. A contractor copies the bottom-right alternative in Figure 3B-24 exactly, using white markings, and omits every accompanying sign because the pavement treatment already tells drivers what to do. Other layout and marking details are correct. Identify what that figure alternative contains, determine whether an outline or cross-hatching must be added, and decide whether the sign omission is permitted. State the minimal correction for the conditions described; there is no railroad crossing at this location.\n\n**Required visual input:** [Figure 3B-24 — original PDF page 624](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=624). Read the specified original diagram; do not infer dimensions by scaling it.",
 "SAMPLE019": "At an established school crossing across a major street, 20 schoolchildren cross during the highest crossing hour. During the 30-minute period in which the children use the crossing, an engineering study counts 24 adequate traffic gaps; adequacy has been assessed for the number and size of the actual groups. The nearest traffic control signal along the major street is 250 feet away. The study specifically finds that a proposed signal at the school crossing will not restrict progressive traffic movement. It considers other remedial measures and separately concludes that a traffic control signal is justified. Determine whether the school-crossing gap and volume criteria are met, whether the 250-foot separation prevents applying this warrant, and whether the new signal must have pedestrian signal heads even if a crossing guard remains. Assess only these issues, not the complete signal layout.",
 "SAMPLE020": "Two permanent static signs use individual LEDs for conspicuity; neither is a changeable message sign or a hand-held paddle. A STOP sign has red LEDs, each 0.20 inch in diameter, entirely within the background strip extending one border width inward from the border. They flash simultaneously at a steady 55 flashes per minute only when a vehicle is detected and remain off otherwise. A diamond-shaped curve warning sign has yellow LEDs of the same diameter in the corresponding background strip; they flash simultaneously at a steady 55 flashes per minute continuously. The LEDs are individually spaced, not clusters; retroreflection, legibility, and other details have been checked. Assess background placement and actuation for each sign, and identify the needed corrections. Can both arrangements be accepted because their diameters, colors, and flash rates are appropriate?",
 "SAMPLE021": "A new railroad entrance gate has a 35-foot arm length measured from the center of the gate mast to the tip. Both sides are fully retroreflective with alternating vertical red and white stripes at 16-inch intervals measured horizontally. On the front, the retroreflective sheeting is 4 inches high for the first 30 feet from the mast center and 2 inches high for the remaining 5 feet; height is measured with the arm down. In the normal operating sequence, with no constant-warning-time or advanced-system exception, train detection and flashing-light activation occur at t = 0 seconds. Downward motion starts at t = 3, the gate reaches horizontal at t = 20, and the train arrives at t = 25. The gate stays down until the train completely clears at t = 40. With no other train detected, it rises from t = 40 to t = 54, and the flashing lights cease immediately after it reaches upright at t = 54. Other gate details are satisfactory. Assess the front sheeting and stated timing under the applicable Standards and Guidance; specify the sheeting correction and distinguish any mandatory timing failure from a recommendation departure."
}

# Later, for the questions kept sealed until the end: put them on Drive as a JSON
# file {id: text}, write its path here, and Run all. Then the DEV cases and those
# are answered (not the samples).
SEALED_FILE = None

if SEALED_FILE:
    ANSWER_SET = json.loads(Path(SEALED_FILE).read_text())
    SET_NAME = "sealed"
else:
    ANSWER_SET, SET_NAME = SAMPLE, "sample"
QUESTIONS = {**{c["case_id"]: c["text"] for c in DEV}, **ANSWER_SET}
DEV_IDS = [c["case_id"] for c in DEV]
CASE = {c["case_id"]: c for c in DEV}
assert not set(DEV_IDS) & set(ANSWER_SET), "a question id is used twice"

RETR_COMMIT = vr.retrieval_commit(REPO_DIR)
STORE_TAG = vr.store_tag(CFG.base_dir)
INSTRUCTIONS_SHA = hashlib.sha256((PARSER_TASK + PARSER_CONTRACT).encode()).hexdigest()
RUNS = Path(CFG.base_dir) / "parser_runs"
KQ_FILE = RUNS / f"kq_r{RETR_COMMIT}_s{STORE_TAG}.json"
RUN_DIR = RUNS / f"e2e_r{RETR_COMMIT}_s{STORE_TAG}_p{INSTRUCTIONS_SHA[:8]}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(DEV_ALL)} DEV cases for the retrieval check")
print(f"{len(QUESTIONS)} questions to answer: {len(DEV)} DEV + {len(ANSWER_SET)} {SET_NAME}")
print("retrieval code:", RETR_COMMIT, "| store:", STORE_TAG, "| parser instructions:", INSTRUCTIONS_SHA[:12])
print("run folder    :", RUN_DIR)

### CELL 5 — retrieval on the 154 DEV cases, before and after

*Before* is the retrieval test of 7 October 2026 (same code until this change), except the 12 cases corrected on 9 October, which are run again here with the new part switched off. *After* is the new retrieval. A case **passes** when every target section, and the source paragraph the case was written from, reach what the parser reads.

For every case that misses now or missed before, it prints where each target came from (the whole question, a sentence, a manual term, a heading) and how each view of the question ranked it. **If fewer cases pass than before, the notebook stops here.** Send me this cell's output either way.

In [ ]:
# CELL 5 — the retrieval check on DEV (no LLM). Saved as it goes.
OUT_DIR = Path(CFG.base_dir) / "retrieval_dev_results"
CHECK_FILE = OUT_DIR / f"retrieval_check_r{RETR_COMMIT}_s{STORE_TAG}.json"
BEFORE_FILE = OUT_DIR / f"retrieval_before_r{RETR_COMMIT}_s{STORE_TAG}.json"

t0 = time.time()
NOW = vr.run_check_resumable(lambda c: rc.run_case(pipeline.retriever, c), DEV_ALL, CHECK_FILE)
print(f"\nnew retrieval: {len(NOW)} cases, median {sorted(r['seconds'] for r in NOW)[len(NOW) // 2]:.1f}s a case")
rc.print_summary(rc.summarize(NOW))

def run_old(case):
    CFG.compile_find_provisions = False
    try:
        return rc.run_case(pipeline.retriever, case)
    finally:
        CFG.compile_find_provisions = True

EARLIER = vr.load_json(Path(CFG.base_dir) / vr.EARLIER_RETRIEVAL_TEST, {})
if EARLIER:
    print(f"\nearlier run: {vr.EARLIER_RETRIEVAL_TEST} | store {EARLIER.get('store_chunks')} items "
          f"(now {n_store}) | commit {EARLIER.get('commit')}")
    if EARLIER.get("store_chunks") not in (None, n_store):
        print("WARNING: the vector store has changed since the earlier run; 'before' is run again here for every case")
        EARLIER = {}
else:
    print("\nno earlier run on Drive; 'before' is run here for every case")
BEFORE = vr.results_before(DEV_ALL, EARLIER.get("results"), run_old, out_path=BEFORE_FILE)

G = rc.gate(NOW, BEFORE)
print("\n=== before -> after (target provisions reach what the parser reads) ===")
rc.print_gate(G)
print("\nper style, before:"); rc.print_summary(rc.summarize(BEFORE))
miss_now = [r["case_id"] for r in NOW if not rc.passes(r)]
miss_before = [r["case_id"] for r in BEFORE if not rc.passes(r)]
show = [c["case_id"] for c in DEV_ALL if c["case_id"] in set(miss_now) | set(miss_before)]
print(f"\n=== {len(show)} cases that miss now or missed before: where each target came from ===")
rc.print_targets(NOW, show)
vr.save_json(CHECK_FILE, {"commit": COMMIT, "retrieval_commit": RETR_COMMIT, "settings": SETTINGS,
                          "store_items": n_store, "summary": rc.summarize(NOW), "gate": G,
                          "results": NOW})
print("\nsaved ->", CHECK_FILE, f"| {time.time() - t0:.0f}s")
assert G["ok"], ("the new retrieval finds the target provisions for FEWER DEV cases than before "
                 f"({G['pass_before']} -> {G['pass_now']}). Stopped here. Send this cell's output.")

In [ ]:
# CELL 6 — retrieval for every question to answer (Kq), saved to Drive.
# The parser reads exactly this. Only questions not yet retrieved (or whose text
# changed) are retrieved.
from mrag.vine.parser import build_parser_prompt
FIG_KEYS = ("figure_id", "caption", "n_sheets", "image_paths", "image_path", "page_pdf", "source")
KQ = vr.load_json(KQ_FILE, {})
changed = [q for q in QUESTIONS if q in KQ and KQ[q].get("question") != QUESTIONS[q]]
missing = [q for q in QUESTIONS if q not in KQ] + changed
print(f"{KQ_FILE.name}: {len(KQ)} questions already retrieved, {len(missing)} to retrieve")
for i, qid in enumerate(missing, 1):
    q = QUESTIONS[qid]
    t0 = time.time()
    res = pipeline.retriever.retrieve_for_compile(q)
    find = res.debug.get("find") or {}
    KQ[qid] = vr.plain({"question": q, "chunks": res.chunks,
                        "figures": [{k: f.get(k) for k in FIG_KEYS if k in f} for f in res.figures],
                        "closure": res.debug.get("closure", {}),
                        "find": {k: find.get(k) for k in ("sentences", "terms", "headings", "pool")},
                        "seconds": round(time.time() - t0, 1)})
    vr.save_json(KQ_FILE, KQ)
    print(f"  {i:2d}/{len(missing)} {qid:26s} Kq {len(res.chunks):3d} items, {len(res.figures)} figures, "
          f"{time.time() - t0:4.1f}s")
for qid in DEV_IDS:
    t = pc.target_check(None, CASE[qid], KQ[qid]["chunks"])
    print(f"  DEV {qid:26s} the provision the case was written from reaches the parser: {t['target_chunk_in_kq']}")
sizes = {qid: len(build_parser_prompt(KQ[qid]["question"], KQ[qid]["chunks"], "", None, KQ[qid]["figures"]))
         for qid in QUESTIONS}
longest = max(sizes, key=sizes.get)
print(f"longest parser prompt: {sizes[longest]:,} characters (about {sizes[longest] // 4:,} tokens)")

In [ ]:
# CELL 7 — which plans are still to be made; free the GPU for the parser if any are.
import gc, torch
import mrag.ask as _mask
from mrag.vine import table_data

PLANS_FILE = RUN_DIR / "plans.json"
PLAN_RECORDS = vr.load_json(PLANS_FILE, {})
# no plan yet, or the last try broke (the model call or the run, not the plan): make it
TO_PARSE = [q for q in QUESTIONS if vr.plan_needs_retry(PLAN_RECORDS.get(q), QUESTIONS[q])]
print(f"plans: {len(QUESTIONS) - len(TO_PARSE)} already made (from {PLANS_FILE.name}), {len(TO_PARSE)} to make")

kg = pipeline.kg                  # the parser's checks read the graph (CPU only)
TABLES = {}
for t in table_data.load(Path(CFG.tables_jsonl)):
    TABLES.setdefault(t.table_id, []).append(t)
TABLE_LIST = [t for ts in TABLES.values() for t in ts]
GPU_UTIL = 0.90
if TO_PARSE:
    vc.stop_servers()             # a model server left from an earlier run still holds the GPU
    # everything that holds the search models (a second Run all in the same session
    # has the checkers' copies too)
    for _name in ("pipeline", "retriever", "VIEW", "probe"):
        globals().pop(_name, None)
    _mask._pipeline = None
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}, {total / 2**30:.1f} GiB, free {free / 2**30:.1f} GiB")
    assert total / 2**30 > 38, "the parser needs an A100 (40 or 80 GB). Runtime > Change runtime type > A100, then Run all."
    assert free >= GPU_UTIL * total, (f"only {free / 2**30:.1f} GiB free; the parser needs {GPU_UTIL * total / 2**30:.1f}. "
                                      "Runtime > Restart session, then Run all (everything done so far is saved).")

In [ ]:
# CELL 8 — the parser's model: vLLM in its own environment, the model on local disk.
# Its own environment so its newer torch/transformers never touch the retrieval
# stack. About 5 minutes to install and 5-10 to download 28 GB. Skipped when
# every plan is already made.
VENV          = "/content/vllm_env"
VLLM_VERSION  = "0.31.0"
MODEL_ID      = "Qwen/Qwen3.8-27B-FP8"      # 27.8 GB; on an A100 vLLM runs it as 8-bit weights
HF_LOCAL      = "/content/hf_local"          # local disk, not Drive
if TO_PARSE:
    # mrag's config points the Hugging Face cache at Drive with HUGGINGFACE_HUB_CACHE,
    # which wins over HF_HOME; every cache variable is set to local disk here.
    for k in ("HF_HOME", "HF_HUB_CACHE", "HUGGINGFACE_HUB_CACHE", "TRANSFORMERS_CACHE"):
        os.environ.pop(k, None)
    os.environ["HF_HOME"] = HF_LOCAL
    os.environ["HF_HUB_CACHE"] = f"{HF_LOCAL}/hub"
    if not os.path.exists(f"{VENV}/bin/vllm"):
        !pip install -q uv
        !uv venv {VENV} --python 3.12 -q
        !uv pip install -q --python {VENV}/bin/python "vllm=={VLLM_VERSION}" --torch-backend=auto
    !{VENV}/bin/python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| CUDA', torch.version.cuda, '| transformers', transformers.__version__)"
    !{VENV}/bin/python -c "from huggingface_hub import snapshot_download as s; print(s('{MODEL_ID}'))"
    SNAPS = Path(HF_LOCAL) / "hub" / ("models--" + MODEL_ID.replace("/", "--")) / "snapshots"
    assert SNAPS.is_dir(), f"the model did not land in {SNAPS} -- send the output above"
    MODEL_PATH = sorted(SNAPS.iterdir(), key=lambda p: p.stat().st_mtime)[-1]
    REVISION = MODEL_PATH.name
    print("model on local disk:", MODEL_PATH, "| revision", REVISION)
else:
    print("every plan is already made -- the parser is not needed")

In [ ]:
# CELL 9 — start the parser's server at effort "xhigh" and check it answers.
# One request at a time, fixed seed, the model card's thinking-mode sampling.
# First start takes 5-15 minutes. Skipped when every plan is already made.
SEED, EFFORT = 42, "xhigh"
ENGINE = f"vllm {VLLM_VERSION}"
if TO_PARSE:
    SERVER = vc.launch(f"{VENV}/bin/vllm", str(MODEL_PATH), log_path="/content/vllm_server.log", port=8000,
                       env={"HF_HUB_OFFLINE": "1",
                            # FlashInfer's sampler is compiled on first use and that fails
                            # on this setup. Seeded requests never use it anyway.
                            "VLLM_USE_FLASHINFER_SAMPLER": "0"},
                       served_name="parser", max_model_len=73728, min_model_len=65536,
                       gpu_memory_utilization=GPU_UTIL, seed=SEED)
    MAX_LEN = SERVER["max_model_len"]
    MAX_TOKENS = min(49152, MAX_LEN - 21504)
    print(f"server ready in {SERVER['seconds']:.0f}s | max_model_len {MAX_LEN} | max_tokens {MAX_TOKENS}")
    probe = vc.make_ask_vllm(SERVER["base_url"], "parser", model_id=MODEL_ID, revision=REVISION,
                             engine=ENGINE, seed=SEED, max_tokens=2048, reasoning_effort="low")
    reply = probe('Reply with this JSON object and nothing else: {"ok": true}', [])
    print("probe reply:", repr(reply[:80]), "|", probe.last["finish_reason"])
    assert '"ok"' in reply, "the server answered, but not with the JSON asked for -- send the output above"

    def make_ask(max_tokens):
        return vc.make_ask_vllm(SERVER["base_url"], "parser", model_id=MODEL_ID, revision=REVISION,
                                engine=ENGINE, seed=SEED, max_tokens=max_tokens, reasoning_effort=EFFORT,
                                cache_dir=str(RUNS / "model_cache"))
    # every prompt gets MAX_TOKENS to think and write, unless it is so long that
    # prompt + MAX_TOKENS would not fit in the server's window (then what fits)
    ASK = vr.BudgetedAsk(make_ask, SERVER["base_url"], "parser", max_len=MAX_LEN, max_tokens=MAX_TOKENS,
                         template_kwargs={"enable_thinking": True, "reasoning_effort": EFFORT})
    n = ASK.count(build_parser_prompt(KQ[longest]["question"], KQ[longest]["chunks"], "", None,
                                      KQ[longest]["figures"]))
    print(f"longest prompt ({longest}): {n['tokens']:,} tokens ({n['how']}); "
          f"room left for thinking and the plan: {MAX_LEN - n['tokens']:,}")
    PARSE_MANIFEST = {"commit": COMMIT, "model": MODEL_ID, "revision": REVISION, "vllm": VLLM_VERSION,
                      "server_command": SERVER["cmd"], "gpu": torch.cuda.get_device_name(0), "seed": SEED,
                      "sampling": vc.QWEN_THINKING, "reasoning_effort": EFFORT, "max_tokens": MAX_TOKENS,
                      "max_model_len": MAX_LEN, "max_attempts": 3, "fallback": False,
                      "instructions": "fix5", "instructions_sha256": INSTRUCTIONS_SHA,
                      "kq_file": KQ_FILE.name, "retrieval_settings": SETTINGS}
    vr.save_json(RUN_DIR / "parse_manifest.json", PARSE_MANIFEST)
else:
    print("every plan is already made -- no server started")

### CELL 10 — the parser writes every plan

DEV first, then the other questions. Each plan is saved on Drive the moment it is made. For each DEV case the whole plan is printed; for the other questions, one line.

Look for: `valid first time` (best), `valid after repair`, `failed`; `finish` should be `stop` (`length` means the model ran out of room while thinking); whether the plan uses the provision the DEV case was written from.

In [ ]:
# CELL 10 — every plan (about 7 minutes a question at xhigh). Saved as it goes.
from mrag.vine.parser import make_semantic_parser
from mrag.vine.compile import instantiate

if TO_PARSE:
    parse = make_semantic_parser(ASK, max_attempts=3, fall_back_to_baseline=False,
                                 check_ref=kg.is_known_citation, tables=TABLES)
order = [q for q in DEV_IDS if q in TO_PARSE] + [q for q in TO_PARSE if q not in DEV_IDS]
for i, qid in enumerate(order, 1):
    v = KQ[qid]
    t0 = time.time()
    try:
        spec, rep = parse(v["question"], v["chunks"], "", v["figures"])
        net = instantiate(spec)[0] if spec is not None else None
        rec = pc.make_record(qid, v["question"], v, spec, rep, net, time.time() - t0, case=CASE.get(qid))
    except Exception as e:                                     # noqa: BLE001
        rec = vr.failed_plan_record(qid, v["question"], f"the parse stopped: {e!r}", kq=v)
    PLAN_RECORDS[qid] = rec
    vr.save_json(PLANS_FILE, PLAN_RECORDS)
    s = rec["stats"]
    print(f"  {i:2d}/{len(order)} {qid:26s} {rec['outcome']:19s} attempts {rec['attempts']} "
          f"checks {s.get('obligations', 0):2d} merges {s.get('merges', 0)} | {rec['seconds']:5.0f}s "
          f"| finish {[c.get('finish_reason') for c in rec['calls']]} "
          f"| prompt tokens {[c.get('prompt_tokens') for c in rec['calls']]}")
if TO_PARSE:
    vc.stop_servers()             # frees the GPU for retrieval during execution

DEV_PLANS = [PLAN_RECORDS[q] for q in DEV_IDS]
print("\n=== DEV plans ===")
pc.print_summary(pc.summarize(DEV_PLANS))
hits = [r["target"].get("source_hit") for r in DEV_PLANS if (r.get("target") or {}).get("target_chunk_in_kq")]
print(f"plan uses the provision the case was written from: {sum(bool(h) for h in hits)}/{len(hits)}")
others = [PLAN_RECORDS[q] for q in QUESTIONS if q not in DEV_IDS]
print(f"{SET_NAME} plans: " + ", ".join(f"{k} {sum(1 for r in others if r['outcome'] == k)}"
                                        for k in ("valid first time", "valid after repair", "failed")))
print()
for r in DEV_PLANS:
    pc.print_plan(r)
    print()

In [ ]:
# CELL 11 — retrieval back on the GPU (the checkers search the manual), and the checker model.
from mrag.vine import anthropic_client as ac
if "pipeline" not in globals():
    pipeline = load_retrieval()
retriever, kg = pipeline.retriever, pipeline.kg
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception as e:
    assert os.environ.get("ANTHROPIC_API_KEY"), (
        "Add ANTHROPIC_API_KEY to Colab's secrets (key icon on the left), allow this notebook "
        f"to use it, then Run all again. ({e!r})")

CHECKER_MODEL, CHECKER_EFFORT, CHECKER_MAX_TOKENS = "claude-sonnet-5-5", "high", 32000
MAX_IMAGES = 4
CHECKER_ASK = ac.make_ask_anthropic(CHECKER_MODEL, effort=CHECKER_EFFORT, max_tokens=CHECKER_MAX_TOKENS,
                                    cache_dir=str(RUNS / "checker_cache"))
reply = CHECKER_ASK('Reply with this JSON object and nothing else: {"ok": true}', [])
print("text probe  :", repr(reply[:80]), "|", CHECKER_ASK.last["finish_reason"], "|", CHECKER_ASK.last["model"])
assert '"ok"' in reply, "the API answered, but not with the JSON asked for -- send the output above"

fig = kg.figure("Figure 2B-1")
assert fig, "Figure 2B-1 is not in the graph -- send the output above"
img = kg.g.nodes[fig].get("image_path") or (kg.g.nodes[fig].get("image_paths") or [""])[0]
assert os.path.exists(img), f"the figure image is not on Drive: {img}"
reply = CHECKER_ASK('Reply with one JSON object and nothing else: {"shape": "<the shape of the first '
                    'sign shown, in one or two words>"}', [img])
print("image probe :", repr(reply[:120]), "|", CHECKER_ASK.last["finish_reason"])

from mrag.vine.lookup import ManualView, Lookups, TOOL_SPECS, tools_digest
VIEW = ManualView.from_retriever(retriever)
assert VIEW is not None, "the labels and lookups need the VINE graph -- send the output above"
probe = Lookups(VIEW, max_lookups=2)
reply, tr = CHECKER_ASK.converse(
    'Use the open_section tool to open Section 1C.01, then reply with one JSON object and nothing '
    'else: {"title": "<the section title>"}', [], probe.specs(), probe.run, max_lookups=2)
print("lookup probe:", repr(reply[:120]), "| lookups", [e["tool"] for e in probe.log],
      "| turns", tr["turns"], "| stopped:", tr["stopped"] or "no")
assert probe.log and '"title"' in reply, "the tool-use conversation did not work -- send the output above"

In [ ]:
# CELL 12 — the plans as networks, the checkers, and one function that executes a plan.
from mrag.vine.compile import NetworkSpec, instantiate
from mrag.vine.run import build_verifiers
from mrag.vine.execute import execute
from mrag.vine.answer import compose
from mrag.vine.model_verifiers import READING_RULES, LOOKUP_RULES

PLANS = {}
for qid in QUESTIONS:
    d = PLAN_RECORDS[qid].get("spec")
    spec = NetworkSpec.from_dict(d) if d else None
    net, problems = instantiate(spec) if spec is not None else (None, ["no plan"])
    PLANS[qid] = {"spec": spec, "net": net, "problems": problems}
print(f"{sum(1 for p in PLANS.values() if p['net'] is not None and not p['problems'])} of {len(PLANS)} plans can be executed")

MODEL_OPTIONS = {"use_pointers": True, "max_pointed_chunks": 8, "max_images": MAX_IMAGES,
                 "labels": True, "use_lookups": True, "max_lookups": 12,
                 "named_exceptions": True, "max_named_exceptions": 6}
EXEC_FILE = RUN_DIR / f"exec_{COMMIT}.json"
EXEC_MANIFEST = {"commit": COMMIT, "plans_file": str(PLANS_FILE), "instructions_sha256": INSTRUCTIONS_SHA,
                 "checker_model": CHECKER_MODEL, "checker_effort": CHECKER_EFFORT,
                 "checker_max_tokens": CHECKER_MAX_TOKENS, "checker_temperature": "not settable (model default)",
                 "model_options": MODEL_OPTIONS, "hand_over": True, "kq_file": KQ_FILE.name,
                 "checker_tools": tools_digest(TOOL_SPECS),
                 "checker_rules_sha256": hashlib.sha256((READING_RULES + LOOKUP_RULES).encode()).hexdigest()}
vr.save_json(RUN_DIR / f"exec_manifest_{COMMIT}.json", EXEC_MANIFEST)

def run_case(qid, ask=None):
    ask = ask or CHECKER_ASK
    q = QUESTIONS[qid]
    plan = PLANS[qid]
    n0 = len(ask.calls)
    t0 = time.time()
    if plan["net"] is None or plan["problems"]:
        return ec.make_exec_record(qid, q, plan["spec"], plan["net"], None, None, [], 0.0, None)
    verifiers = build_verifiers(tables=TABLE_LIST, kg=kg, retriever=retriever, ask=ask,
                                query=q, hand_over=True, model_options=MODEL_OPTIONS)
    trace = execute(plan["net"], verifiers)
    answer = compose(q, plan["net"], trace)
    rec = ec.make_exec_record(qid, q, plan["spec"], plan["net"], trace, answer,
                              ask.calls[n0:], time.time() - t0, None)
    rec["cost_usd"] = ac.cost_usd(ask.calls[n0:], CHECKER_MODEL)
    return rec
print("checker:", CHECKER_MODEL, "| effort", CHECKER_EFFORT, "| results ->", EXEC_FILE)

### CELL 13 — execute every plan

DEV first. Each executed question is saved on Drive the moment it is done; a question already done is not run again. Every checker reply is saved too, so a re-run replays it.

For each DEV case the whole execution is printed: each check's status and who decided it, the lookups (`looked`), manual text it found itself (`found`), exceptions handed to it (`handed`), and the decision. The other questions are executed here but shown only in CELL 14.

In [ ]:
# CELL 13 — execute every plan (the checkers), then DEV in full. Saved as it goes.
# an earlier result counts only if it executed this question's current plan
EXEC = {r["case_id"]: r for r in vr.load_json(EXEC_FILE, {}).get("records", [])
        if r["case_id"] in QUESTIONS and r.get("question") == QUESTIONS[r["case_id"]]
        and vr.same_json(r.get("spec"), PLAN_RECORDS[r["case_id"]].get("spec")) and not r.get("error")}
todo = [q for q in DEV_IDS if q not in EXEC] + [q for q in QUESTIONS if q not in DEV_IDS and q not in EXEC]
print(f"{len(EXEC)} already executed (from {EXEC_FILE.name}), {len(todo)} to execute")
for i, qid in enumerate(todo, 1):
    try:
        r = run_case(qid)
    except Exception as e:                                     # noqa: BLE001
        r = {"case_id": qid, "question": QUESTIONS[qid], "spec": PLAN_RECORDS[qid].get("spec"),
             "certificates": [], "waves": [], "terminal": None, "answer": None, "calls": [],
             "seconds": 0.0, "error": repr(e)}
    EXEC[qid] = r
    vr.save_json(EXEC_FILE, {"manifest": EXEC_MANIFEST, "records": [EXEC[q] for q in QUESTIONS if q in EXEC]})
    who = "DEV" if qid in CASE else SET_NAME
    print(f"  {i:2d}/{len(todo)} {who:6s} {qid:26s} decision {str(r.get('terminal')):8s} "
          f"checks {sum(1 for x in r['certificates'] if x['kind'] == 'check'):2d} "
          f"lookups {sum(len(x.get('lookups') or []) for x in r['certificates']):3d} "
          f"| {r['seconds']:5.0f}s | ${r.get('cost_usd', 0):.3f}" + (f" | ERROR {r['error']}" if r.get("error") else ""))

DEV_EXEC = [EXEC[q] for q in DEV_IDS]
print("\n=== DEV ===")
ec.print_exec_summary(ec.summarize_execution([r for r in DEV_EXEC if not r.get("error")]))
print(f"cost of new checker calls this session: ${ac.cost_usd(CHECKER_ASK.calls, CHECKER_MODEL):.3f}\n")
for r in DEV_EXEC:
    if r.get("error"):
        print("=" * 110); print(r["case_id"], "ERROR:", r["error"]); continue
    ec.print_execution(r)
    print(f"cost: ${r.get('cost_usd', 0):.4f}\n")

In [ ]:
# CELL 14 — the answers to the other questions, for scoring outside the pipeline,
# and one file with the whole run. Both are downloaded.
OTHER = [EXEC[q] for q in QUESTIONS if q not in DEV_IDS]
print(f"=== {SET_NAME} questions: {len(OTHER)} ===")
vr.print_answers(OTHER)
stamp = time.strftime("%Y%m%d_%H%M")
ANSWERS_FILE = RUN_DIR / f"answers_{SET_NAME}_{stamp}.txt"
ANSWERS_FILE.write_text("\n".join(vr.answer_line(r) for r in OTHER) + "\n")
RESULT_FILE = RUN_DIR / f"vine_run_{stamp}.json"
vr.save_json(RESULT_FILE, {
    "commit": COMMIT, "retrieval_commit": RETR_COMMIT, "store": STORE_TAG,
    "retrieval_settings": SETTINGS, "retrieval_check": {"file": CHECK_FILE.name, "gate": G},
    "parse_manifest": vr.load_json(RUN_DIR / "parse_manifest.json", {}), "exec_manifest": EXEC_MANIFEST,
    "plans": {q: PLAN_RECORDS[q] for q in QUESTIONS}, "executions": {q: EXEC[q] for q in QUESTIONS}})
print("\nsaved ->", ANSWERS_FILE)
print("saved ->", RESULT_FILE)
from google.colab import files
files.download(str(ANSWERS_FILE))
files.download(str(RESULT_FILE))